# Lifespan Predictor — SageMaker Training Demo

Train a DeepSurv survival model on AWS SageMaker using the NASA Turbofan Engine dataset. The dataset contains sensor readings from aircraft engines; the model predicts Remaining Useful Life (RUL).

This notebook covers:
1. Data loading and preparation
2. Model configuration and SageMaker training
3. Evaluation metrics, calibration curves, and training plots

## 1. Setup

**Prerequisites** — set these environment variables before running:

```bash
export SAGEMAKER_ALGORITHM_ARN='arn:aws:sagemaker:eu-west-1:926554707418:algorithm/lifespan-predictor-1-0-1'
export SAGEMAKER_ROLE_ARN='arn:aws:iam::ACCOUNT:role/service-role/YOUR-SageMaker-ExecutionRole'
```

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lifespan_predictor_sdk import SurvivalModel, download_nasa_dataset, get_nasa_train_dataframe, get_nasa_test_dataframe

### Load Data

In [ ]:
data_dir = "data/nasa"
download_nasa_dataset(data_dir)

train_df = get_nasa_train_dataframe(data_dir, datasets=[1])
test_df, test_labels_df = get_nasa_test_dataframe(data_dir, datasets=[1])

print(f"Training data: {len(train_df['unit'].unique())} units, {len(train_df)} rows")
print(f"Test data: {len(test_df['unit'].unique())} units, {len(test_df)} rows")
train_df.head()

## 2. Train on SageMaker

Configure a DeepSurv model and submit a SageMaker training job. The SDK handles data upload, job submission, and result download automatically.

In [ ]:
model = SurvivalModel({
    "model_type": "deepsurv",
    "algorithm_arn": os.environ["SAGEMAKER_ALGORITHM_ARN"],
    "instance_type": "ml.m5.large",
    "num_nodes": [24],
    "epochs": 30,
    "rolling_means": [5, 10, 20, 30],
    "max_rul": 125,
    "calibration": "isotonic",
})
model.print_config()

In [ ]:
model.fit(train_df, val_split=0.2, test_df=test_df, test_labels_df=test_labels_df)

## 3. Results

### Evaluation Metrics

In [ ]:
model.get_evaluation_result().report()

### Training History

In [ ]:
model.get_training_log().plot()

### View Job Details

In [ ]:
model.print_training_job_summary()

### Deploy Model (Optional)

In [ ]:
model.deploy()

---

## Summary

This notebook demonstrated SageMaker training with the `SurvivalModel` wrapper.

### What's Handled Automatically

- Uploads train and test dataframes to S3
- Submits Sagemaker training job
- Waits for job completion
- SDK makes it easy to fetch model artifact, training logs, and evaluation metrics

### Next Steps

- Use your own custom dataset
- Leverage builtin hyperparameter tuning (see examples/hyperparameter_optimisation.ipynb)